## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [3]:
MODEL = "SmolLM2-135M-Instruct-Q4_K_M"
DB_NAME = "vector_db"
load_dotenv(override=True)

False

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [10]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, base_url="http://127.0.0.1:8080/v1", model_name=MODEL, api_key="none")

### These LangChain objects implement the method `invoke()`

In [12]:
retriever.invoke("Who is Avery?")

[Document(id='b0460ce8-64d4-49b9-877e-c28d09833574', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [13]:
llm.invoke("Who is Avery?")

AIMessage(content='Avery is a fictional character from the Marvel Cinematic Universe (MCU). She is a superhero who appears in the Marvel Cinematic Universe\'s "Avengers" series. Avery is a skilled fighter and has been featured in various Marvel films, including "Iron Man" and "Avengers: Endgame." She is also a member of the Avengers\' "Guardians of the Galaxy" team.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 35, 'total_tokens': 118, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': '/home/imphe1218/Models/SmolLM2-135M-Instruct-Q4_K_M.gguf', 'system_fingerprint': 'b10098-0278d8362', 'id': 'chatcmpl-72ncevKCd3vLqFBBfLeZthCHDU0shzVF', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--de3ab0a2-11b5-4818-a2df-f0865ea35109-0', usage_metadata={'input_tokens': 35, 'output_tokens': 83, 'total_tokens': 118, 'input_token_de

## Time to put this together!

In [14]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [15]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [16]:
answer_question("Who is Averi Lancaster?", [])

"Averi Lancaster is a highly respected and accomplished AI engineer who has made significant contributions to the Insurellm company. She has been a key player in the company's early stages, driving innovation and driving growth.\n\nAveri is known for her exceptional problem-solving skills, strategic thinking, and ability to balance customer needs with business objectives. She has been instrumental in driving the company's growth and success, and has been recognized as a key contributor to the company's success in the industry.\n\nAveri has been involved in several key projects, including the development of the Insurellm InsureTech platform, which has been a major contributor to the company's growth and success. She has also been involved in strategic data initiatives, such as the development of the Insurellm InsureTech platform, which has been a major contributor to the company's growth and success.\n\nAveri's work has been recognized by industry leaders and has been featured in numero

## What could possibly come next? 😂

In [17]:
gr.ChatInterface(answer_question).launch()

/home/imphe1218/Projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!